In [41]:
from abc import ABC, abstractmethod

In [42]:
class Runnable(ABC):

    @abstractmethod
    def invoke(self, input_data):
        pass

In [43]:
import random

class NakliLLM(Runnable): 

    def __init__(self):
        print('LLM start')

    def invoke(self, prompt):
        response_list = [
            'Mumbai is Capital of Maharashtra',
            'Hero ISL is a Foodball leage',
            'AI stand for Artificial Intelligence'
        ]

        return {"response": random.choice(response_list)}

    def predict(self, prompt):
        response_list = [
            'Mumbai is Capital of Maharashtra',
            'Hero ISL is a Foodball leage',
            'AI stand for Artificial Intelligence'
        ]

        return {"response": random.choice(response_list)}

In [44]:
class NakliPromptTemplate(Runnable):

    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict):
        if isinstance(input_dict, dict):
            return self.template.format(**input_dict)
        elif isinstance(input_dict, str) and self.input_variables:
            return self.template.format(**{self.input_variables[0]: input_dict})
        return self.template

    def format(self, input_dict):
        return self.invoke(input_dict)

In [45]:
class NakliStrOutParser(Runnable):
    def __init__(self):
        pass
    def invoke(self, input_data):
        if isinstance(input_data, dict) and 'response' in input_data:
            return input_data['response']
        return str(input_data)

In [46]:
class RunnableConnector(Runnable):

    def __init__(self, runnable_list):
        self.runnable_list = list(runnable_list)

    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
        return input_data

In [47]:
template = NakliPromptTemplate(
    template = 'Write a {length} poem about {topic}',
    input_variables = ['length', 'topic']
) 

In [48]:
llm = NakliLLM()

LLM start


In [12]:
parser = NakliStrOutParser()

In [49]:
chain = RunnableConnector([template, llm])

In [50]:
chain.invoke({'length':'long', 'topic':'india'})

{'response': 'Hero ISL is a Foodball leage'}

In [51]:
template1 = NakliPromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

In [52]:
template2 = NakliPromptTemplate(
    template = 'Explain the following joke {response}',
    input_variables = ['response']
)

In [53]:
llm = NakliLLM()

LLM start


In [54]:
parser = NakliStrOutParser()

In [55]:
chain1 = RunnableConnector([template1, llm])


In [56]:
chain1.invoke({'topic':'AI'})

{'response': 'AI stand for Artificial Intelligence'}

In [57]:
chain2 = RunnableConnector([template2, llm, parser])


In [58]:
final_chain = RunnableConnector([chain1, chain2])

In [59]:
final_chain.invoke({'topic':'cricket'})

'AI stand for Artificial Intelligence'